# 본 자료는 TTS 커널을 이용해 주세요.

In [1]:
pip install -U "transformers==4.41.2" "tokenizers>=0.15,<0.20" accelerate sentencepiece safetensors


Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install  transformers accelerate torch fastapi uvicorn python-multipart soundfile librosa

Note: you may need to restart the kernel to use updated packages.


In [3]:
import transformers, TTS
print("transformers:", transformers.__version__)
print("TTS:", TTS.__version__)


transformers: 4.41.2
TTS: 0.22.0


In [4]:
import os
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

from pathlib import Path
import pandas as pd
from TTS.api import TTS

ROOT = Path.home() / "data_team_tts"
CSV_PATH = ROOT / "refs.csv"
AUDIO_ROOT = ROOT / "audio"
OUT_DIR = Path("./tts_out")
OUT_DIR.mkdir(exist_ok=True, parents=True)

MODEL_NAME = "tts_models/multilingual/multi-dataset/xtts_v2"
SPEAKER_ID = "student01"  # 기본 화자

# 1) refs.csv 로드
df = pd.read_csv(CSV_PATH, keep_default_na=False, na_filter=False)
df["file"] = df["file"].astype(str).str.strip()
df["transcript"] = df["transcript"].astype(str).str.strip()

# 2) 특정화자 필터링
df_spk = df[df["file"].str.startswith(f"{SPEAKER_ID}/")].copy()

# 3) wav path 구성 + 존재 파일만
df_spk["wav_path"] = df_spk["file"].apply(lambda p: (AUDIO_ROOT / p).resolve())
df_spk = df_spk[df_spk["wav_path"].apply(lambda p: Path(p).exists())].copy()
df_spk = df_spk.sort_values("file")

SPEAKER_WAVS_DEFAULT = [str(p) for p in df_spk["wav_path"].tolist()[:5]]  # 예: 5개만
print(f"[{SPEAKER_ID}] wav count:", len(SPEAKER_WAVS_DEFAULT))

# 4) 모델 로딩 (1회)
tts = TTS(MODEL_NAME, gpu=True)
print("TTS loaded:", MODEL_NAME)

def xtts_to_file(text: str, speaker_wavs, out_path, **overrides):
    kw = dict(
        speaker_wav=speaker_wavs,
        language="ko",
        temperature=0.1,
        top_p=0.6,
        top_k=50,
        repetition_penalty=5.0,
        length_penalty=1.0,
        split_sentences=False,
    )
    kw.update(overrides)

    out_path = str(out_path)
    tts.tts_to_file(text=text, file_path=out_path, **kw)
    return out_path

[student01] wav count: 5
 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.


/home/data/bootcamp/user2/venvs/tts/lib/python3.10/site-packages/TTS/api.py:70: UserWarning: `gpu` will be deprecated. Please use `tts.to(device)` instead.
  warnings.warn("`gpu` will be deprecated. Please use `tts.to(device)` instead.")


 > Using model: xtts


/home/data/bootcamp/user2/venvs/tts/lib/python3.10/site-packages/TTS/tts/layers/xtts/xtts_manager.py:5: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  self.speakers = torch.load(speaker_file_path)
/home/data/bootcamp/user2/venvs/tts/lib/python3.10/site-packages/TTS/utils/io.py:54: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  return torch.load(f, map_location=map_location, **kwargs)


TTS loaded: tts_models/multilingual/multi-dataset/xtts_v2


In [5]:
import nest_asyncio
nest_asyncio.apply()

import os, time, uuid, logging
from pathlib import Path
from fastapi import FastAPI, Form

app = FastAPI(title="Notebook XTTS API")

# ---- logging (ASR에서 쓰던 방식 그대로 사용 가능하게) ----
WIDGET_HANDLER_NAME = "widget_out_handler"

class OutputWidgetHandler(logging.Handler):
    def __init__(self, out_widget):
        super().__init__()
        self.out_widget = out_widget
        self.name = WIDGET_HANDLER_NAME

    def emit(self, record):
        try:
            msg = self.format(record)
            self.out_widget.append_stdout(msg + "\n")
        except Exception:
            pass

def hook_logs_to_widget(out_widget):
    handler = OutputWidgetHandler(out_widget)
    handler.setLevel(logging.INFO)
    handler.setFormatter(logging.Formatter("[%(levelname)s] %(name)s: %(message)s"))

    for lname in ["fastapi", "uvicorn", "uvicorn.error", "uvicorn.access"]:
        lg = logging.getLogger(lname)
        lg.setLevel(logging.INFO)
        lg.propagate = False
        lg.handlers = []
        lg.addHandler(handler)

log = logging.getLogger("fastapi")

@app.get("/health")
def health():
    log.info("[HEALTH] ok")
    return {
        "ok": True,
        "model": MODEL_NAME,
        "speaker_id_default": SPEAKER_ID,
        "n_refs_default": len(SPEAKER_WAVS_DEFAULT),
        "out_dir": str(OUT_DIR.resolve()),
    }

@app.post("/tts")
def tts_api(
    text: str = Form(...),
    # 아래는 옵션(수업용: 필요할 때만 사용)
    temperature: float = Form(0.1),
    top_p: float = Form(0.6),
    top_k: int = Form(50),
    repetition_penalty: float = Form(5.0),
    length_penalty: float = Form(1.0),
):
    text = (text or "").strip()
    if not text:
        return {"ok": False, "error": "text is empty"}

    # 파일명은 충돌 방지용 uuid
    wav_name = f"xtts_{SPEAKER_ID}_{uuid.uuid4().hex[:8]}.wav"
    out_wav = OUT_DIR / wav_name

    log.info(f"[TTS] request: text_len={len(text)} temp={temperature} top_p={top_p} top_k={top_k}")
    t0 = time.time()

    xtts_to_file(
        text,
        SPEAKER_WAVS_DEFAULT,
        out_wav,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repetition_penalty=repetition_penalty,
        length_penalty=length_penalty,
    )

    dt_ms = int((time.time() - t0) * 1000)
    log.info(f"[TTS] done: ms={dt_ms} wav={wav_name}")

    return {
        "ok": True,
        "wav_name": wav_name,
        "wav_path": str(out_wav.resolve()),
        "elapsed_ms": dt_ms,
    }


In [6]:
import threading, asyncio, time
import ipywidgets as widgets
from IPython.display import display
import uvicorn

out = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="8px", height="280px", overflow="auto"))
port_w = widgets.IntText(value=8003, description="Port:", layout=widgets.Layout(width="200px"))
host_w = widgets.Text(value="0.0.0.0", description="Host:", layout=widgets.Layout(width="260px"))

start_btn = widgets.Button(description="Start", button_style="success")
stop_btn  = widgets.Button(description="Stop", button_style="danger")
clear_btn = widgets.Button(description="Clear")

status = widgets.HTML(value="<b>Status:</b> stopped")

display(widgets.HBox([host_w, port_w, start_btn, stop_btn, clear_btn, status]), out)

def ui_log(msg: str):
    out.append_stdout(msg + "\n")

_server = None
_thread = None

def start_server(_=None):
    global _server, _thread
    if _thread and _thread.is_alive():
        ui_log("[INFO] Server already running.")
        status.value = "<b>Status:</b> running"
        return

    hook_logs_to_widget(out)

    host = host_w.value.strip()
    port = int(port_w.value)

    config = uvicorn.Config(app, host=host, port=port, log_level="info", access_log=True, log_config=None)
    _server = uvicorn.Server(config)

    def runner():
        try:
            ui_log(f"[INFO] Starting server on http://{host}:{port}")
            asyncio.run(_server.serve())
            ui_log("[INFO] Server exited.")
        except Exception as e:
            ui_log(f"[ERROR] {type(e).__name__}: {e}")

    _thread = threading.Thread(target=runner, daemon=True)
    _thread.start()

    time.sleep(0.3)
    status.value = "<b>Status:</b> running"
    ui_log("[INFO] Ready. POST /tts")

def stop_server(_=None):
    global _server, _thread
    if not _thread or not _thread.is_alive():
        ui_log("[INFO] Server is not running.")
        status.value = "<b>Status:</b> stopped"
        return

    ui_log("[INFO] Stopping server...")
    _server.should_exit = True
    _thread.join(timeout=2.0)

    status.value = "<b>Status:</b> stopped"
    ui_log("[INFO] Server stopped.")
    _server = None
    _thread = None

def clear_logs(_=None):
    out.clear_output()
    ui_log("[INFO] Logs cleared.")

start_btn.on_click(start_server)
stop_btn.on_click(stop_server)
clear_btn.on_click(clear_logs)

ui_log("[INFO] UI ready. Click Start.")


Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…